In [1]:
import os

print("Current path:", os.getcwd())
print("Files:", os.listdir())

Current path: C:\Users\vilan\OneDrive\Desktop\EE4216_Biscuit_Detection
Files: ['.ipynb_checkpoints', 'EE4216_Biscuit_Detection.ipynb', 'input_images', 'output_images']


In [2]:
import os

input_folder = "input_images"
print(os.listdir(input_folder))

['image1.jpeg', 'image10.jpeg', 'image11.jpeg', 'image12.jpeg', 'image13.jpeg', 'image14.jpeg', 'image15.jpeg', 'image16.jpeg', 'image2.jpeg', 'image3.jpeg', 'image4.jpeg', 'image5.jpeg', 'image6.jpeg', 'image7.jpeg', 'image8.jpeg', 'image9.jpeg']


In [4]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread("input_images/test.jpeg")

if img is None:
    print("❌ Image not found")
else:
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    plt.axis("off")

❌ Image not found


In [5]:
import os
print(os.getcwd())


C:\Users\vilan\OneDrive\Desktop\EE4216_Biscuit_Detection


In [6]:
import os
print(os.listdir("input_images"))

['image1.jpeg', 'image10.jpeg', 'image11.jpeg', 'image12.jpeg', 'image13.jpeg', 'image14.jpeg', 'image15.jpeg', 'image16.jpeg', 'image2.jpeg', 'image3.jpeg', 'image4.jpeg', 'image5.jpeg', 'image6.jpeg', 'image7.jpeg', 'image8.jpeg', 'image9.jpeg']


In [7]:
import os
import cv2

input_folder = "input_images"

files = os.listdir(input_folder)
print("Files found:", files)

for file in files:
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        path = os.path.join(input_folder, file)
        print("Trying:", path)

        img = cv2.imread(path)

        if img is None:
            print("❌ Failed to load:", file)
        else:
            print("✅ Successfully loaded:", file)

Files found: ['image1.jpeg', 'image10.jpeg', 'image11.jpeg', 'image12.jpeg', 'image13.jpeg', 'image14.jpeg', 'image15.jpeg', 'image16.jpeg', 'image2.jpeg', 'image3.jpeg', 'image4.jpeg', 'image5.jpeg', 'image6.jpeg', 'image7.jpeg', 'image8.jpeg', 'image9.jpeg']
Trying: input_images\image1.jpeg
✅ Successfully loaded: image1.jpeg
Trying: input_images\image10.jpeg
✅ Successfully loaded: image10.jpeg
Trying: input_images\image11.jpeg
✅ Successfully loaded: image11.jpeg
Trying: input_images\image12.jpeg
✅ Successfully loaded: image12.jpeg
Trying: input_images\image13.jpeg
✅ Successfully loaded: image13.jpeg
Trying: input_images\image14.jpeg
✅ Successfully loaded: image14.jpeg
Trying: input_images\image15.jpeg
✅ Successfully loaded: image15.jpeg
Trying: input_images\image16.jpeg
✅ Successfully loaded: image16.jpeg
Trying: input_images\image2.jpeg
✅ Successfully loaded: image2.jpeg
Trying: input_images\image3.jpeg
✅ Successfully loaded: image3.jpeg
Trying: input_images\image4.jpeg
✅ Successful

In [26]:
import cv2
import numpy as np
import os

# ==============================
# PATH SETUP
# ==============================
input_folder = "input_images"
output_folder = "output_images"

os.makedirs(output_folder, exist_ok=True)

image_files = [
    f for f in os.listdir(input_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Found images:", image_files)

# ==============================
# PROCESS IMAGES
# ==============================
for file_name in image_files:

    image_path = os.path.join(input_folder, file_name)
    image = cv2.imread(image_path)

    if image is None:
        continue

    output = image.copy()
    img_area = image.shape[0] * image.shape[1]

    # ==============================
    # SEGMENTATION (simple & stable)
    # ==============================
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    lower = np.array([10, 40, 40])
    upper = np.array([35, 255, 255])

    mask = cv2.inRange(hsv, lower, upper)

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # ==============================
    # REMOVE BACKGROUND
    # ==============================
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    clean_mask = np.zeros_like(mask)

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if 2000 < area < img_area * 0.15:
            cv2.drawContours(clean_mask, [cnt], -1, 255, -1)

    mask = clean_mask

    # ==============================
    # FINAL CONTOURS
    # ==============================
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    intact = 0
    broken = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)
        if area < 2000:
            continue

        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

        # ==============================
        # CONVEXITY DEFECTS (MAIN LOGIC)
        # ==============================
        hull = cv2.convexHull(cnt, returnPoints=False)

        defect_count = 0

        if hull is not None and len(hull) > 3:
            defects = cv2.convexityDefects(cnt, hull)

            if defects is not None:
                for i in range(defects.shape[0]):
                    s, e, f, d = defects[i][0]

                    # depth of defect (IMPORTANT)
                    if d > 2000:   # threshold tuned for biscuits
                        defect_count += 1

        # ==============================
        # SHAPE FEATURES (backup)
        # ==============================
        circularity = 4 * np.pi * area / (perimeter * perimeter)

        rect = cv2.minAreaRect(cnt)
        width, height = rect[1]

        if width == 0 or height == 0:
            continue

        aspect_ratio = max(width, height) / min(width, height)

        # ==============================
        # FINAL CLASSIFICATION
        # ==============================

        # 🔴 Broken: has concave defects
        if defect_count >= 1:
            label = "Broken Biscuit"
            color = (0, 0, 255)
            broken += 1

        # 🟢 Intact circular
        elif circularity > 0.80:
            label = "Intact Biscuit"
            color = (0, 255, 0)
            intact += 1

        # 🟢 Intact square
        elif aspect_ratio < 1.35:
            label = "Intact Biscuit"
            color = (0, 255, 0)
            intact += 1

        else:
            label = "Broken Biscuit"
            color = (0, 0, 255)
            broken += 1

        # ==============================
        # DRAW
        # ==============================
        box = cv2.boxPoints(rect)
        box = np.int32(box)

        x, y, w, h = cv2.boundingRect(cnt)

        cv2.drawContours(output, [cnt], -1, color, 2)
        cv2.drawContours(output, [box], 0, color, 2)

        cv2.putText(output, label, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # ==============================
    # DISPLAY COUNT
    # ==============================
    cv2.putText(output, f"Intact: {intact}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.putText(output, f"Broken: {broken}", (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    # ==============================
    # SAVE
    # ==============================
    save_path = os.path.join(output_folder, "result_" + file_name)
    cv2.imwrite(save_path, output)

    print(f"{file_name} -> Intact: {intact}, Broken: {broken}")

print("Done! Check output_images folder.")

Found images: ['image1.jpeg', 'image10.jpeg', 'image11.jpeg', 'image12.jpeg', 'image13.jpeg', 'image14.jpeg', 'image15.jpeg', 'image16.jpeg', 'image2.jpeg', 'image3.jpeg', 'image4.jpeg', 'image5.jpeg', 'image6.jpeg', 'image7.jpeg', 'image8.jpeg', 'image9.jpeg']
image1.jpeg -> Intact: 3, Broken: 6
image10.jpeg -> Intact: 3, Broken: 5
image11.jpeg -> Intact: 3, Broken: 5
image12.jpeg -> Intact: 2, Broken: 6
image13.jpeg -> Intact: 3, Broken: 5
image14.jpeg -> Intact: 3, Broken: 5
image15.jpeg -> Intact: 3, Broken: 5
image16.jpeg -> Intact: 3, Broken: 5
image2.jpeg -> Intact: 3, Broken: 6
image3.jpeg -> Intact: 3, Broken: 6
image4.jpeg -> Intact: 3, Broken: 6
image5.jpeg -> Intact: 3, Broken: 6
image6.jpeg -> Intact: 3, Broken: 6
image7.jpeg -> Intact: 3, Broken: 6
image8.jpeg -> Intact: 3, Broken: 6
image9.jpeg -> Intact: 3, Broken: 5
Done! Check output_images folder.
